In [2]:
# imports
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import resnet18, ResNet18_Weights

In [5]:
class MultiTaskModel(nn.Module):
  def __init__(self, num_age_classes=5, num_gender_classes=2, num_emotion_classes=7):
    super(MultiTaskModel, self).__init__()

    # Load Pre-Trained Model
    self.backbone = resnet18(weights=ResNet18_Weights.DEFAULT)

    # Remove the fully connected layer
    in_features = self.backbone.fc.in_features
    self.backbone.fc = nn.Identity()

    # Heads
    self.age_head = nn.Sequential(
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, num_age_classes)
    )
    self.gender_head = nn.Sequential(
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, num_gender_classes)
    )
    self.emotion_head = nn.Sequential(
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, num_emotion_classes)
    )

  def forward(self, x):
    features = self.backbone(x)

    age_out = self.age_head(features)
    gender_out = self.gender_head(features)
    emotion_out = self.emotion_head(features)

    return age_out, gender_out, emotion_out


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


# Load the Model

In [6]:
model = MultiTaskModel().to(device)
model.load_state_dict(torch.load("/content/best_model.pth"))
print('model loaded succesfully')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 135MB/s]


RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory

# Define the transformers

In [5]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Labels for mapping

In [6]:
age_classes = ["Child", "Teen", "Young Adult", "Adult", "Senior"]
gender_classes = ["Male", "Female"]
emotion_classes = ["Angry", "Disgust", "Fear", "Happy", "Neutral", "Sad", "Surprise"]

In [11]:
from PIL import Image
img_path = '/content/20-male-happy.jpeg'
img = Image.open(img_path).convert('RGB')
transform(img).shape


torch.Size([3, 224, 224])

In [17]:
import torch

def predict_image(model, img_path, transform, device):
  model.eval()

  img = Image.open(img_path).convert('RGB')
  img = transform(img).unsqueeze(0).to(device)

  with torch.no_grad():
    age_out, gender_out, emotion_out = model(img)

  age_pred = torch.argmax(age_out, dim=1).item()
  gender_pred = torch.argmax(gender_out, dim=1).item()
  emotion_pred = torch.argmax(emotion_out, dim=1).item()

  return (
      age_classes[age_pred],
      gender_classes[gender_pred],
      emotion_classes[emotion_pred]
  )



In [18]:
age, gender, emotion = predict_image(model, img_path, transform, device)

print(f"Age Group: {age}")
print(f"Gender: {gender}")
print(f"Emotion: {emotion}")

Age Group: Adult
Gender: Male
Emotion: Angry
